In [1]:
import os
import sys
import importlib
from dotenv import load_dotenv

# If site-packages was patched during this notebook session, reload key memengine modules
# so newly created memory objects use the updated code.
RELOAD_ORDER = [
    "memengine.function.Judge",
    "memengine.function",
    "memengine.operation.Recall",
    "memengine.operation.Store",
    "memengine.memory.GAMemory",
    "memengine.memory.SCMemory",
    "memengine.memory.MGMemory",
    "memengine.memory.RFMemory",
    "memengine.memory.MTMemory",
    "memengine",
]
for module_name in RELOAD_ORDER:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

import memengine

load_dotenv()

OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Missing env: OPENROUTER_API_KEY"
assert OPENROUTER_BASE_URL, "Missing env: OPENROUTER_BASE_URL"
assert OPENROUTER_MODEL, "Missing env: OPENROUTER_MODEL"

from memengine import (
    MemoryConfig,
    FUMemory,
    STMemory,
    LTMemory,
    MBMemory,
    GAMemory,
    SCMemory,
    MGMemory,
    RFMemory,
    MTMemory,
)

d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 测试 LLM 是否可连接 (OpenRouter) ---
def test_llm_connection():
    from openai import OpenAI
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
    resp = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        temperature=0.0,
        max_tokens=16,
    )
    content = (resp.choices[0].message.content or "").strip()
    return content

try:
    reply = test_llm_connection()
    print("LLM 连接成功. 模型:", OPENROUTER_MODEL)
    print("回复:", reply)
except Exception as e:
    print("LLM 连接失败:", type(e).__name__, str(e))

LLM 连接成功. 模型: mistralai/mistral-nemo
回复: OK


In [5]:
import json
from pathlib import Path

DATA_PATH = Path("locomo10.json")
assert DATA_PATH.exists(), f"Not found: {DATA_PATH.resolve()}"

raw = json.loads(DATA_PATH.read_text(encoding="utf-8"))
print("items:", len(raw))
print("keys of first item:", list(raw[0].keys()))

qa = raw[0]["qa"]
print("qa count:", len(qa))
print("sample qa[0] keys:", list(qa[0].keys()))

# Turn qa pairs into memory observations (plain text)
# (Some entries may miss the 'answer' field; we skip those.)
qa_valid = [x for x in qa if "answer" in x]
observations = [
    f"Q: {x['question']}\nA: {x['answer']}\nEvidence: {', '.join(x.get('evidence', []))}"
    for x in qa_valid
]

# Pick one query to test recall (use an existing question)
query = qa_valid[0]["question"]
print("query:", query)
print("observation[0]:\n", observations[0])

items: 10
keys of first item: ['qa', 'conversation', 'event_summary', 'observation', 'session_summary', 'sample_id']
qa count: 199
sample qa[0] keys: ['question', 'answer', 'evidence', 'category']
query: When did Caroline go to the LGBTQ support group?
observation[0]:
 Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3


In [6]:
def make_common_config(*, usable_gpu: str = "", display_method: str = "ScreenDisplay") -> dict:
    # Minimal, runnable configs derived from the official open-source defaults,
    # but adapted to run locally without external model paths.
    return {
        "global_config": {"usable_gpu": usable_gpu},
        "storage": {},
        "display": {
            "method": display_method,
            "prefix": "----- Current Memory Start (%s) -----",
            "suffix": "----- Current Memory End -----",
            "key_format": "(%s)",
            "key_value_sep": "\n",
            "item_sep": "\n",
            # FileDisplay-only arg (ignored by ScreenDisplay)
            "output_path": "logs/sample.log",
        },
        "recall": {
            "truncation": {
                "method": "LMTruncation",
                "mode": "word",
                "number": 256,
                "path": "",  # only used for token-based truncation
            },
            "utilization": {
                "method": "ConcateUtilization",
                "prefix": "[Memory Start]",
                "suffix": "[Memory End]",
                "list_config": {"index": True, "sep": "\n"},
                "dict_config": {"key_format": "(%s)", "key_value_sep": "\n", "item_sep": "\n"},
            },
            "empty_memory": "None",
        },
        "store": {},
    }


def make_text_retrieval_config(*, topk: int = 5, st_model: str = "sentence-transformers/all-MiniLM-L6-v2") -> dict:
    # LTMemory / MBMemory rely on embeddings; this uses SentenceTransformers.
    return {
        "method": "TextRetrieval",
        "encoder": {
            "method": "STEncoder",
            "name": st_model,
            "dimension": 384,
            "path": st_model,
        },
        "mode": "cosine",
        "topk": topk,
    }


def run_memory(
    memory,
    obs_list,
    query_text,
    *,
    with_time: bool = False,
    fixed_time: int | None = None,
    time_bucket: int = 1,
):
    memory.reset()
    for i, obs in enumerate(obs_list):
        if with_time:
            if fixed_time is not None:
                t = fixed_time
            else:
                tb = max(1, int(time_bucket))
                t = i // tb
            memory.store({"text": obs, "time": t})
        else:
            memory.store(obs)
    return memory.recall(query_text)

In [7]:
# --- FUMemory (Full / long-context) ---
fu_cfg = make_common_config()
fu_cfg["name"] = "FUMemory"
fu_cfg["store"] = {"method": "FUMemoryStore"}
fu_cfg["recall"]["method"] = "FUMemoryRecall"

fu = FUMemory(MemoryConfig(fu_cfg))
fu_ans = run_memory(fu, observations[:30], query)
print("FUMemory recall result:\n", fu_ans)

# Optional: visualize internal storage
fu.display()

FUMemory recall result:
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
[2] Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
[3] Q: What did Caroline research?
A: Adoption agencies
Evidence: D2:8
[4] Q: What is Caroline's identity?
A: Transgender woman
Evidence: D1:5
[5] Q: When did Melanie run a charity race?
A: The sunday before 25 May 2023
Evidence: D2:1
[6] Q: When is Melanie planning on going camping?
A: June 2023
Evidence: D2:7
[7] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[8] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[9] Q: When did Caroline meet up with her friends, family, and mentors?
A: The week before 9 June 2023
Evidence: D3:11
[10] Q: How long has Caroline had her current group of friends for?

In [8]:
# --- STMemory (Short-term / recent window) ---
st_cfg = make_common_config()
st_cfg["name"] = "STMemory"
st_cfg["store"] = {"method": "LTMemoryStore"}
st_cfg["recall"].update({
    "method": "STMemoryRecall",
    "time_retrieval": {"method": "TimeRetrieval", "mode": "raw", "topk": 5},
})

st = STMemory(MemoryConfig(st_cfg))
st_ans = run_memory(st, observations[:30], query)
print("STMemory recall result:\n", st_ans)

st.display()

STMemory recall result:
 [Memory Start]
[0] Q: When did Melanie go to the pottery workshop?
A: The Friday before 15 July 2023
Evidence: D8:2
[1] Q: When did Caroline go to the adoption meeting?
A: The friday before 15 July 2023
Evidence: D8:9
[2] Q: Would Caroline pursue writing as a career option?
A: LIkely no; though she likes reading, she wants to be a counselor
Evidence: D7:5, D7:9
[3] Q: When did Melanie read the book "nothing is impossible"?
A: 2022
Evidence: D7:8
[4] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[Memory End]
----- Current Memory Start (30) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
counter_id: 0
[Memory Entity 1]
text: Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
counter_id: 1
[Memory Entity 2]
text: Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
co

In [9]:
# --- LTMemory (Long-term / embedding retrieval) ---
lt_cfg = make_common_config()
lt_cfg["name"] = "LTMemory"
lt_cfg["store"] = {"method": "LTMemoryStore"}
lt_cfg["recall"].update({
    "method": "LTMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
})

lt = LTMemory(MemoryConfig(lt_cfg))
lt_ans = run_memory(lt, observations[:80], query)
print("LTMemory recall result:\n", lt_ans)

lt.display()

LTMemory recall result:
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[2] Q: What LGBTQ+ events has Caroline participated in?
A: Pride parade, school speech, support group
Evidence: D5:1, D8:17, D3:1, D1:3
[3] Q: In what ways is Caroline participating in the LGBTQ community?
A: Joining activist group, going to pride parades, participating in an art show, mentoring program
Evidence: D10:3, D5:1, D9:12, D9:2
[4] Q: What career path has Caroline decided to persue?
A: counseling or mental health for Transgender people
Evidence: D4:13, D1:11
[Memory End]
----- Current Memory Start (80) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
counter_id: 0
[Memory Entity 1]
text: Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
counter_id: 1
[Memory Entity 2]
text: Q: What 

In [10]:
# --- MBMemory (MemoryBank) ---
# MBMemory will summarize when the `time` bucket changes.
# In this notebook we DO want to trigger summarization, but we avoid summarizing on every item
# by bucketing times (e.g., one summary per 20 observations).

mb_cfg = make_common_config()
mb_cfg["name"] = "MBMemory"
mb_cfg["store"] = {
    "method": "MBMemoryStore",
    "summarizer": {
        "method": "LLMSummarizer",
        "LLM_config": {
            "method": "APILLM",
            "name": OPENROUTER_MODEL,
            "api_key": OPENROUTER_API_KEY,
            "base_url": OPENROUTER_BASE_URL,
            "temperature": 0.0,
        },
        "prompt": {
            "template": "Content: {content}\nSummarize the above content concisely, extracting the main themes and key information.",
            "input_variables": ["content"],
        },
    },
}
mb_cfg["recall"].update({
    "method": "MBMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
    # omit "forget" in this basic demo to make results deterministic/visible
})

mb = MBMemory(MemoryConfig(mb_cfg))

# OpenRouter/free-tier models can occasionally return empty/None content.
# MBMemory's pipeline assumes summarizer always returns a non-empty string;
# if it returns None, embedding will crash.
#
# Important: MBMemoryStore.reset() expects `summarizer` to have a `.reset()` method,
# so we wrap it in a small callable object rather than replacing it with a bare function.

class SafeSummarizer:
    def __init__(self, inner):
        self.inner = inner

    def reset(self):
        r = getattr(self.inner, "reset", None)
        if callable(r):
            r()

    def __call__(self, content):
        for _ in range(2):
            try:
                s = self.inner(content)
            except Exception:
                s = None
            if isinstance(s, str) and s.strip():
                return s

        text = content if isinstance(content, str) else str(content)
        text = text.strip()
        if not text:
            return "Summary (fallback): <empty content>"
        return "Summary (fallback): " + (text[:600] + ("..." if len(text) > 600 else ""))

mb.store_op.summarizer = SafeSummarizer(mb.store_op.summarizer)

mb_ans = run_memory(mb, observations[:80], query, with_time=True, time_bucket=20)
print("MBMemory recall result:\n", mb_ans)

mb.display()

MBMemory recall result:
 [Memory Start]
[0] None
[1] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[2] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[3] Q: What LGBTQ+ events has Caroline participated in?
A: Pride parade, school speech, support group
Evidence: D5:1, D8:17, D3:1, D1:3
[4] Q: In what ways is Caroline participating in the LGBTQ community?
A: Joining activist group, going to pride parades, participating in an art show, mentoring program
Evidence: D10:3, D5:1, D9:12, D9:2
[5] Q: What career path has Caroline decided to persue?
A: counseling or mental health for Transgender people
Evidence: D4:13, D1:11
[6] Q: What transgender-specific events has Caroline attended?
A: Poetry reading, conference
Evidence: D17:19, D15:13
[7] Q: When did Caroline join a new activist group?
A: The Tuesday before 20 July 2023
Evidence: D10:3
[8] Q: When is Caroline going to the transgender conference?
A: July 2023
Evidence: D5:1

In [11]:
# --- GAMemory (Generative Agents) ---
# NOTE: GAMemory judges an importance score for every stored observation (LLM call).
# To keep this demo lightweight, we store only a small subset.

ga_cfg = make_common_config()
ga_cfg["name"] = "GAMemory"

ga_cfg["store"] = {"method": "GAMemoryStore"}

ga_cfg["recall"].update(
    {
        "method": "GAMemoryRecall",
        "topk": 8,
        "text_retrieval": make_text_retrieval_config(topk=32),
        "time_retrieval": {
            "method": "TimeRetrieval",
            "mode": "exp",
            "coef": {"decay": 0.99},
            "topk": 32,
        },
        "importance_retrieval": {"method": "ValueRetrieval", "mode": "identical", "topk": 32},
        "importance_judge": {
            "method": "LLMJudge",
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "post_scale": 10,
            "prompt": {
                "template": "You are scoring how important a memory is for future use.\nMemory: {message}\nReturn ONLY a number from 0 to 10.",
                "input_variables": ["message"],
            },
        },
    }
)

# Reflection is available via ga.manage('reflect'), but we keep it effectively disabled here
# by setting an extremely high threshold to avoid extra LLM calls / strict formatting asserts.
ga_cfg["reflect"] = {
        "reflector": {
            "threshold": 1_000_000_000,
            "reflection_topk": 5,
            "question_number": 3,
            "insight_number": 3,
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "question_prompt": {
                "template": "Given the information below, write exactly {question_number} questions (one per line).\n\nInformation:\n{information}",
                "input_variables": ["information", "question_number"],
            },
            "insight_prompt": {
                "template": "Given the statements below, write exactly {insight_number} high-level insights (one per line).\n\nStatements:\n{statements}",
                "input_variables": ["statements", "insight_number"],
            },
        }
}

ga = GAMemory(MemoryConfig(ga_cfg))
ga_ans = run_memory(ga, observations[:12], query)
print("GAMemory recall result:\n", ga_ans)

ga.display()

GAMemory recall result:
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: What is Caroline's identity?
A: Transgender woman
Evidence: D1:5
[2] Q: How long has Caroline had her current group of friends for?
A: 4 years
Evidence: D3:13
[3] Q: When did Caroline meet up with her friends, family, and mentors?
A: The week before 9 June 2023
Evidence: D3:11
[4] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[5] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[6] Q: Where did Caroline move from 4 years ago?
A: Sweden
Evidence: D3:13, D4:3
[7] Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
[Memory End]
----- Current Memory Start (12) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
source: False
counter_id:

In [12]:
# --- SCMemory (Self-Controlled Memory) ---
# SCMemory uses LLM components for summarization + control judges.
# Keep the demo subset small.

sc_cfg = make_common_config()
sc_cfg["name"] = "SCMemory"

sc_cfg["store"] = {"method": "SCMemoryStore"}
sc_cfg["recall"].update(
    {
        "method": "SCMemoryRecall",
        "flash_capacity": 5,
        "activation_topk": 8,
        "text_retrieval": make_text_retrieval_config(topk=32),
        "time_retrieval": {
            "method": "TimeRetrieval",
            "mode": "exp",
            "coef": {"decay": 0.99},
            "topk": 32,
        },
        "summarizer": {
            "method": "LLMSummarizer",
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "prompt": {
                "template": "Summarize in 1 sentence, keep key entities and dates.\nContent: {content}",
                "input_variables": ["content"],
            },
        },
        "activation_judge": {
            "method": "LLMJudge",
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "prompt": {
                "template": "Query: {query}\nFlash memory: {flash_memory}\nDo we need to retrieve more history beyond flash memory to answer? Return ONLY True or False.",
                "input_variables": ["query", "flash_memory"],
            },
        },
        "summary_judge": {
            "method": "LLMJudge",
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "prompt": {
                "template": "Query: {query}\nActivation summary: {activation_summary}\nFlash memory: {flash_memory}\nIs the activation summary sufficient (no need for full texts)? Return ONLY True or False.",
                "input_variables": ["query", "activation_summary", "flash_memory"],
            },
        },
    }
)

sc = SCMemory(MemoryConfig(sc_cfg))
sc_ans = run_memory(sc, observations[:12], query)
print("SCMemory recall result:\n", sc_ans)

sc.display()

SCMemory recall result:
 [Memory Start]
[0] Q: Where did Caroline move from 4 years ago?
A: Sweden
Evidence: D3:13, D4:3
[1] Q: How long has Caroline had her current group of friends for?
A: 4 years
Evidence: D3:13
[2] Q: When did Caroline meet up with her friends, family, and mentors?
A: The week before 9 June 2023
Evidence: D3:11
[3] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[4] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[Memory End]
----- Current Memory Start (12) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
time: 0
summary: Caroline attended the LGBTQ support group on **7 May 2023** (D1:3).
counter_id: 0
[Memory Entity 1]
text: Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
time: 1
summary: Melanie painted a sunrise in 2022, as evidenced by D1:12.
counter_id: 1
[Memory Entity 2]
text: Q: What fields wou

In [13]:
# --- MGMemory (MemGPT-style hierarchical memory) ---
# MGMemory is LLM-heavy (trigger + summarizer). This cell provides a minimal runnable config.
# For a real demo, you'd tune the trigger prompt + function list.

mg_cfg = make_common_config()
mg_cfg["name"] = "MGMemory"

mg_cfg["store"] = {
    "method": "MGMemoryStore",
    "summarizer": {
        "method": "LLMSummarizer",
        "LLM_config": {
            "method": "APILLM",
            "name": OPENROUTER_MODEL,
            "api_key": OPENROUTER_API_KEY,
            "base_url": OPENROUTER_BASE_URL,
            "temperature": 0.0,
        },
        "prompt": {
            "template": "Recursive summary (keep factual details).\nPrevious summary: {recursive_summary}\nNew content: {flush_context}\nReturn updated summary only.",
            "input_variables": ["recursive_summary", "flush_context"],
        },
    },
    # flush_checker only needs .check_truncation_needed(text); LMTruncation provides it.
    "flush_checker": {"method": "LMTruncation", "mode": "word", "number": 120, "path": ""},
}

mg_cfg["recall"].update(
    {
        "method": "MGMemoryRecall",
        "warning_threshold": 0.8,
        "warning_content": "[Warning: memory capacity is near the limit]",
        "recall_retrieval": make_text_retrieval_config(topk=5),
        "archival_retrieval": make_text_retrieval_config(topk=5),
        "trigger": {
            "method": "LLMTrigger",
            "LLM_config": {
                "method": "APILLM",
                "name": OPENROUTER_MODEL,
                "api_key": OPENROUTER_API_KEY,
                "base_url": OPENROUTER_BASE_URL,
                "temperature": 0.0,
            },
            "func_list": [
                {
                    "name": "memory_recall",
                    "args": ["query"],
                    "args_type": ["str"],
                    "func_description": "Retrieve related items from recall storage into FIFO memory.",
                    "args_description": ["query: retrieval query"],
                },
                {
                    "name": "memory_retrieval",
                    "args": ["query"],
                    "args_type": ["str"],
                    "func_description": "Retrieve related items from archival storage into working memory.",
                    "args_description": ["query: retrieval query"],
                },
                {
                    "name": "memory_transfer",
                    "args": ["memory_list"],
                    "args_type": ["list"],
                    "func_description": "Transfer items from FIFO memory to working memory.",
                    "args_description": ["memory_list: list of FIFO indexes"],
                },
                {
                    "name": "memory_archive",
                    "args": ["memory_list"],
                    "args_type": ["list"],
                    "func_description": "Archive items from FIFO memory into recall storage.",
                    "args_description": ["memory_list: list of FIFO indexes"],
                },
                {
                    "name": "memory_save",
                    "args": ["memory_list"],
                    "args_type": ["list"],
                    "func_description": "Save items from working memory into archival storage.",
                    "args_description": ["memory_list: list of working memory indexes"],
                },
            ],
            "few_shot": "",
            "prompt": {
                "template": "You are managing a memory OS.\n{warning_content}{no_execute_prompt}\n{function_prompt}\n\nMemory state:\n{memory_prompt}\n\nUser text:\n{text}\n\nReturn ONE OR MORE function calls, one per line (e.g. memory_recall(\"Alice\")).\nIf no function is needed, return: NO_EXECUTE",
                "input_variables": [
                    "warning_content",
                    "no_execute_prompt",
                    "function_prompt",
                    "few_shot",
                    "memory_prompt",
                    "text",
                ],
            },
            "no_execuate": "NO_EXECUTE",
        },
    }
)

mg = MGMemory(MemoryConfig(mg_cfg))
mg_ans = run_memory(mg, observations[:20], query)
print("MGMemory recall result:\n", mg_ans)

mg.display()

Successfully execute Function [memory_recall(['When did Caroline go to the LGBTQ support group?'])]
MGMemory recall result:
 [Memory Start]
(Working Memory)
None
(Recursive Memory Summary)
None
(FIFO Memory)
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
[2] Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
[3] Q: What did Caroline research?
A: Adoption agencies
Evidence: D2:8
[4] Q: What is Caroline's identity?
A: Transgender woman
Evidence: D1:5
[5] Q: When did Melanie run a charity race?
A: The sunday before 25 May 2023
Evidence: D2:1
[6] Q: When is Melanie planning on going camping?
A: June 2023
Evidence: D2:7
[7] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[8] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[9] Q: When did Ca

In [14]:
# --- RFMemory (Reflexion) ---
# RFMemory exposes an optimization step: rf.optimize(new_trial=...) to update a global insight.

rf_cfg = make_common_config()
rf_cfg["name"] = "RFMemory"

rf_cfg["store"] = {"method": "FUMemoryStore"}
rf_cfg["recall"].update({"method": "RFMemoryRecall"})
rf_cfg["optimize"] = {
    "reflector": {
        "method": "TrialReflector",
        "LLM_config": {
            "method": "APILLM",
            "name": OPENROUTER_MODEL,
            "api_key": OPENROUTER_API_KEY,
            "base_url": OPENROUTER_BASE_URL,
            "temperature": 0.0,
        },
        "example": "Trial: I forgot the user's constraint.\nInsight: Always restate constraints before proposing actions.",
        "prompt": {
            "template": "You improve an agent by writing a single concise insight.\nPrevious insight: {previous_insight}\nNew trial:\n{new_trial}\nExample:\n{example}\nReturn ONLY the new insight.",
            "input_variables": ["previous_insight", "new_trial", "example"],
        },
    }
}

rf = RFMemory(MemoryConfig(rf_cfg))

# Store a few items, then recall.
rf_ans = run_memory(rf, observations[:12], query)
print("RFMemory recall result (before optimize):\n", rf_ans)

# Run one optimize step (optional) to populate the global insight.
rf.optimize(new_trial="User asked for a date; I answered with unrelated context.")
rf_ans2 = rf.recall(query)
print("RFMemory recall result (after optimize):\n", rf_ans2)

rf.display()

RFMemory recall result (before optimize):
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
[2] Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
[3] Q: What did Caroline research?
A: Adoption agencies
Evidence: D2:8
[4] Q: What is Caroline's identity?
A: Transgender woman
Evidence: D1:5
[5] Q: When did Melanie run a charity race?
A: The sunday before 25 May 2023
Evidence: D2:1
[6] Q: When is Melanie planning on going camping?
A: June 2023
Evidence: D2:7
[7] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[8] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[9] Q: When did Caroline meet up with her friends, family, and mentors?
A: The week before 9 June 2023
Evidence: D3:11
[10] Q: How long has Caroline had her current gro

In [15]:
# --- MTMemory (MemTree) ---
# MTMemory organizes memory into a tree (GraphStorage) and may summarize internal nodes.

mt_cfg = make_common_config()
mt_cfg["name"] = "MTMemory"

mt_cfg["store"] = {
    "method": "MTMemoryStore",
    "traverse_base_threshold": 0.15,
    "traverse_rate": 1.0,
    "summarizer": {
        "method": "LLMSummarizer",
        "LLM_config": {
            "method": "APILLM",
            "name": OPENROUTER_MODEL,
            "api_key": OPENROUTER_API_KEY,
            "base_url": OPENROUTER_BASE_URL,
            "temperature": 0.0,
        },
        "prompt": {
            "template": "You are updating a parent node summary in a memory tree.\nCurrent content: {current_content}\nNew child content: {new_content}\nThere are {n_children} children.\nReturn an updated concise summary.",
            "input_variables": ["n_children", "new_content", "current_content"],
        },
    },
}

mt_cfg["recall"].update(
    {
        "method": "LTMemoryRecall",
        "text_retrieval": make_text_retrieval_config(topk=8),
    }
)

mt = MTMemory(MemoryConfig(mt_cfg))
mt_ans = run_memory(mt, observations[:25], query)
print("MTMemory recall result:\n", mt_ans)

mt.display()

MTMemory recall result:
 [Memory Start]
[0] Caroline has had her current group of friends for 4 years; her 18th birthday was 10 years ago. She is pursuing a career in counseling or mental health for transgender people, though she likely would not have chosen this path without support growing up (D4:15, D3:5). She attended a transgender conference in July 2023 (D5:13) and had a picnic the week before 6 July 2023 (D6:11).
[1] Q: What career path has Caroline decided to pursue?
A: counseling or mental health for Transgender people
Evidence: D4:13, D1:11, D6:11
[2] Q: When did Melanie sign up for a pottery class?
A: 2 July 2023
Evidence: D5:4
[3] Updated concise summary:

Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14

Q: How long has Caroline had her current group of friends for?
A: 4 years
Evidence: D3:13

Q: Where did Caroline move from 4 years ago?
A: Sweden
Evidence: D3:13, D4:3

Q: How long ago was Caroline's 18th birthday?
A: 10 years ago
Evidence: D4:5
